# Lesson 18: Creating Local AI Agents with Ollama & Qwen2.5

In this notebook, you will:
1. Configure an OpenAI-compatible local endpoint (`http://localhost:11434/v1`) via `Ollama` (`qwen2.5:7b` / `qwen2.5:3b`) with an automatic zero-wait local fallback.
2. Build a **Local Documentation Reviewer** with sandboxed `read_file`, `find_todos`, on-disk `ChromaDB` RAG, and Hybrid Local/Cloud routing.

In [ ]:
!pip install -q -U chromadb openai python-dotenv

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("./sample_project").resolve()
PROJECT_ROOT.mkdir(exist_ok=True)
(PROJECT_ROOT / "auth.py").write_text("# TODO: rotate Ed25519 keys\ndef verify(): return True\n")

def read_file(path: str) -> str:
    full = (PROJECT_ROOT / path).resolve()
    if PROJECT_ROOT not in full.parents and full != PROJECT_ROOT:
        return "Access denied: outside sandbox."
    return full.read_text()

def find_todos() -> list[str]:
    hits = []
    for f in PROJECT_ROOT.glob("*.py"):
        for i, line in enumerate(f.read_text().splitlines(), 1):
            if "TODO" in line or "FIXME" in line:
                hits.append(f"{f.name}:{i}: {line.strip()}")
    return hits

print("Sandboxed read:", read_file("auth.py"))
print("Traversal test:", read_file("../../etc/passwd"))
print("TODOs:", find_todos())